# samples 폴더 전체 파싱 실습

- 목표: `sandbox/W5/Mon/samples` 안의 **모든 파일**을 형식별 파서로 읽어 `ParsedDoc`(블록 목록) 한 가지 모양으로 맞추기
- 형식: pdf · docx · hwpx (규정 문서) + md · txt · html · pptx · xlsx · 스캔 pdf (`_formats`)
- pdf / docx 는 `app/rag/local_parsers.py` 의 `parse_pdf`, `parse_docx` 를 그대로 사용
- 나머지는 이 노트북에서 직접 만든다. docx·hwpx·pptx·xlsx 는 모두 **ZIP 안에 XML** 이 든 구조라 `zipfile` + `lxml` 만으로 읽을 수 있다

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "Mon" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "W5" / "Mon"
SAMPLES = SANDBOX / "samples"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

## 1. 파일 목록 훑어보기

In [ ]:
from collections import Counter

FILES = sorted(p for p in SAMPLES.rglob('*') if p.is_file())
counts = Counter(p.suffix.lower() for p in FILES)

print(f'파일 {len(FILES)}개')
for ext, n in counts.most_common():
    print(f'  {ext:<6} {n}개')

## 2. 공통 결과 모양

- `ParsedBlock(kind, locator, text)` : 덩어리 하나 (`kind` = 조항 / 표)
- `ParsedDoc(blocks, page_count, table_count)` : 문서 한 건
- 표는 마크다운 표 한 덩어리로 만든다 (`_table_to_markdown`)

In [ ]:
import importlib
import zipfile

from lxml import etree

import app.rag.local_parsers as lp
importlib.reload(lp)   # local_parsers.py 를 고쳤다면 다시 읽어오기

from app.integrations.ports import ParsedBlock, ParsedDoc

parse_pdf = lp.parse_pdf
parse_docx = lp.parse_docx
_table_to_markdown = lp._table_to_markdown

print(_table_to_markdown(['버전', '구분'], [['v2.0', '개정']]))

## 3. PDF · DOCX → local_parsers 그대로 사용

In [ ]:
PDF = next(SAMPLES.glob('DOC-HR-014_*_v2.0.pdf'))
DOCX = next(SAMPLES.glob('DOC-HR-014_*_v2.0.docx'))

for path, parser in [(PDF, parse_pdf), (DOCX, parse_docx)]:
    doc = parser(path)
    print(f'{path.name}')
    print(f'  블록 {len(doc.blocks)} · page_count {doc.page_count} · table_count {doc.table_count}')

## 4. HWPX (한글)

- ZIP 안의 `Contents/section0.xml`, `section1.xml` ... 에 본문이 있다
- 문단 `<hp:p>` 안의 글자 `<hp:t>`, 표 `<hp:tbl>` → 행 `<hp:tr>` → 칸 `<hp:tc>`
- 표도 문단(`hp:p`) 안에 들어 있으므로, 문단 글자를 모을 때 **표 안의 글자는 빼야** 중복되지 않는다

In [ ]:
HWPX = next(SAMPLES.glob('DOC-HR-014_*_v2.0.hwpx'))

with zipfile.ZipFile(HWPX) as zf:
    sections = sorted(n for n in zf.namelist() if n.startswith('Contents/section'))
print('섹션 파일 :', sections)

In [ ]:
HP = {'hp': 'http://www.hancom.co.kr/hwpml/2011/paragraph'}

# HWPX를 본문 순서 그대로 읽어 ParsedDoc 으로 리턴
def parse_hwpx(path: pathlib.Path) -> ParsedDoc:
    blocks: list[ParsedBlock] = []
    tables = 0

    with zipfile.ZipFile(path) as zf:
        sections = sorted(n for n in zf.namelist() if n.startswith('Contents/section'))
        for name in sections:
            root = etree.fromstring(zf.read(name))
            for p in root.xpath('./hp:p', namespaces=HP):   # 섹션 바로 아래 문단만 (순서 유지)
                # 1) 표 밖의 글자만 모아 문단 한 덩어리
                text = ''.join(p.xpath('.//hp:t[not(ancestor::hp:tbl)]/text()', namespaces=HP)).strip()
                if text:
                    blocks.append(ParsedBlock('조항', f'{path.stem}', text))
                # 2) 이 문단에 붙어 있는 표 (표 안의 표는 제외)
                for tbl in p.xpath('.//hp:tbl[not(ancestor::hp:tbl)]', namespaces=HP):
                    rows = [
                        [''.join(tc.xpath('.//hp:t/text()', namespaces=HP)).strip() for tc in tr.xpath('./hp:tc', namespaces=HP)]
                        for tr in tbl.xpath('./hp:tr', namespaces=HP)
                    ]
                    if not rows:
                        continue
                    tables += 1
                    blocks.append(ParsedBlock('표', f'표{tables}', _table_to_markdown(rows[0], rows[1:])))

    return ParsedDoc(blocks=blocks, page_count=1, table_count=tables)   # hwpx도 페이지 개념 없음 → 1

doc = parse_hwpx(HWPX)
print(f'블록 {len(doc.blocks)} · table_count {doc.table_count}')
print()
print(next(b for b in doc.blocks if b.kind == '표').text)

## 5. 같은 문서, 세 가지 형식 비교 (DOC-HR-014 v2.0)

- 같은 내용이라도 형식마다 블록 수·표 인식이 다르다 → 어떤 형식을 우선 쓸지 정하는 근거

In [ ]:
for path, parser in [(PDF, parse_pdf), (DOCX, parse_docx), (HWPX, parse_hwpx)]:
    d = parser(path)
    chars = sum(len(b.text) for b in d.blocks)
    print(f'{path.suffix:<6} 블록 {len(d.blocks):>3} · 표 {d.table_count:>2} · {chars:>6,}자')

## 6. 텍스트 계열 : md · txt

- md : `#` 제목이 나올 때마다 새 덩어리
- txt : 빈 줄로 문단 나누기

In [ ]:
import re

def parse_md(path: pathlib.Path) -> ParsedDoc:
    text = path.read_text(encoding='utf-8')
    # 줄 맨 앞의 # 제목 직전에서 자르기 (제목은 다음 덩어리에 붙는다)
    parts = [s.strip() for s in re.split(r'\n(?=#{1,6} )', text) if s.strip()]
    blocks = [ParsedBlock('조항', f'섹션{i}', s) for i, s in enumerate(parts, 1)]
    return ParsedDoc(blocks=blocks, page_count=1, table_count=0)

def parse_txt(path: pathlib.Path) -> ParsedDoc:
    text = path.read_text(encoding='utf-8')
    parts = [s.strip() for s in re.split(r'\n\s*\n', text) if s.strip()]
    blocks = [ParsedBlock('조항', f'문단{i}', s) for i, s in enumerate(parts, 1)]
    return ParsedDoc(blocks=blocks, page_count=1, table_count=0)

for b in parse_md(SAMPLES / '_formats' / 'sample_guide.md').blocks:
    print(f'[{b.locator}] {b.text!r}')
print()
for b in parse_txt(SAMPLES / '_formats' / 'sample_memo.txt').blocks:
    print(f'[{b.locator}] {b.text!r}')

## 7. HTML

- 제목(h1~h6) · 문단(p) · 목록(li) 은 글 덩어리, `<table>` 은 표 덩어리
- 표 안의 p 등은 표에서 한 번에 처리하므로 건너뛴다

In [ ]:
import lxml.html

TEXT_TAGS = {'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p', 'li'}

def parse_html(path: pathlib.Path) -> ParsedDoc:
    tree = lxml.html.fromstring(path.read_bytes())
    blocks: list[ParsedBlock] = []
    tables = 0

    for el in tree.iter():
        if not isinstance(el.tag, str):          # 주석 등은 건너뛰기
            continue
        if el.tag == 'table':
            rows = [[c.text_content().strip() for c in tr.xpath('./th|./td')] for tr in el.xpath('.//tr')]
            if rows:
                tables += 1
                blocks.append(ParsedBlock('표', f'표{tables}', _table_to_markdown(rows[0], rows[1:])))
        elif el.tag in TEXT_TAGS and not el.xpath('ancestor::table'):
            text = el.text_content().strip()
            if text:
                blocks.append(ParsedBlock('조항', el.tag, text))

    return ParsedDoc(blocks=blocks, page_count=1, table_count=tables)

for b in parse_html(SAMPLES / '_formats' / 'sample_page.html').blocks:
    print(f'[{b.kind} · {b.locator}]')
    print(b.text)

## 8. PPTX

- `ppt/slides/slide1.xml`, `slide2.xml` ... 슬라이드마다 파일 하나
- 글자는 `<a:p>` (문단) 안의 `<a:t>` → 슬라이드 하나 = 덩어리 하나
- 주의: 파일 이름을 그냥 정렬하면 slide10 이 slide2 보다 앞에 온다 → 숫자로 정렬

In [ ]:
A = {'a': 'http://schemas.openxmlformats.org/drawingml/2006/main'}

def parse_pptx(path: pathlib.Path) -> ParsedDoc:
    blocks: list[ParsedBlock] = []
    with zipfile.ZipFile(path) as zf:
        slides = [n for n in zf.namelist() if re.fullmatch(r'ppt/slides/slide\d+\.xml', n)]
        slides.sort(key=lambda n: int(re.search(r'(\d+)', n.rsplit('/', 1)[-1]).group(1)))
        for no, name in enumerate(slides, 1):
            root = etree.fromstring(zf.read(name))
            lines = [''.join(p.xpath('.//a:t/text()', namespaces=A)).strip() for p in root.xpath('.//a:p', namespaces=A)]
            text = '\n'.join(l for l in lines if l)
            if text:
                blocks.append(ParsedBlock('조항', f'슬라이드{no}', text))
    return ParsedDoc(blocks=blocks, page_count=len(slides), table_count=0)

for b in parse_pptx(SAMPLES / '_formats' / 'sample_notice.pptx').blocks:
    print(f'[{b.locator}]')
    print(b.text)

## 9. XLSX

- `xl/workbook.xml` : 시트 이름 목록, `xl/worksheets/sheet1.xml` : 셀 값
- 셀 `<c t="...">` 의 종류
  - `inlineStr` : 글자가 셀 안에 바로 (`<is><t>`)
  - `s` : 공유 문자열표(`xl/sharedStrings.xml`)의 번호
  - 그 밖 : 숫자 (`<v>`)
- 시트 하나 = 표 덩어리 하나

In [ ]:
X = {'x': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}

# 'B3' -> 1 (A=0, B=1, ... Z=25, AA=26)
def _col_index(ref: str) -> int:
    n = 0
    for ch in re.match(r'[A-Z]+', ref).group():
        n = n * 26 + (ord(ch) - ord('A') + 1)
    return n - 1

def parse_xlsx(path: pathlib.Path) -> ParsedDoc:
    blocks: list[ParsedBlock] = []
    with zipfile.ZipFile(path) as zf:
        names = zf.namelist()
        shared = []
        if 'xl/sharedStrings.xml' in names:
            sst = etree.fromstring(zf.read('xl/sharedStrings.xml'))
            shared = [''.join(si.xpath('.//x:t/text()', namespaces=X)) for si in sst.xpath('./x:si', namespaces=X)]

        wb = etree.fromstring(zf.read('xl/workbook.xml'))
        sheet_names = wb.xpath('.//x:sheet/@name', namespaces=X)

        # 시트 순서와 sheetN.xml 번호가 같다고 가정 (간단 버전)
        for no, sheet_name in enumerate(sheet_names, 1):
            ws = etree.fromstring(zf.read(f'xl/worksheets/sheet{no}.xml'))
            rows = []
            for row in ws.xpath('.//x:sheetData/x:row', namespaces=X):
                values: dict[int, str] = {}
                for c in row.xpath('./x:c', namespaces=X):
                    kind = c.get('t')
                    if kind == 'inlineStr':
                        v = ''.join(c.xpath('.//x:t/text()', namespaces=X))
                    elif kind == 's':
                        v = shared[int(c.xpath('./x:v/text()', namespaces=X)[0])]
                    else:
                        v = ''.join(c.xpath('./x:v/text()', namespaces=X))
                    values[_col_index(c.get('r'))] = v.strip()
                if values:
                    rows.append([values.get(i, '') for i in range(max(values) + 1)])
            if rows:
                width = max(len(r) for r in rows)
                rows = [r + [''] * (width - len(r)) for r in rows]   # 칸 수 맞추기
                blocks.append(ParsedBlock('표', f'시트:{sheet_name}', _table_to_markdown(rows[0], rows[1:])))

    return ParsedDoc(blocks=blocks, page_count=len(sheet_names), table_count=len(blocks))

for b in parse_xlsx(SAMPLES / '_formats' / 'sample_table.xlsx').blocks:
    print(f'[{b.locator}]')
    print(b.text)

## 10. 스캔 PDF

- 글자 대신 **이미지**만 들어 있어서 `extract_text()` 가 빈 문자열 → 블록 0개
- 파싱 "실패"가 아니라 **OCR 이 필요한 문서**로 따로 표시해야 한다

In [ ]:
SCAN = SAMPLES / '_formats' / 'sample_scanned.pdf'
scan = parse_pdf(SCAN)
print(f'page_count {scan.page_count} · 블록 {len(scan.blocks)}개')
if scan.page_count and not scan.blocks:
    print('→ 텍스트 없음: OCR 대상')

## 11. 전부 한 번에 : 확장자 → 파서 연결표

- `PARSERS` 에 없는 확장자(`.yaml` 등)는 건너뛰기
- 파서 하나가 실패해도 나머지는 계속 돌도록 `try / except`

In [ ]:
PARSERS = {
    '.pdf': parse_pdf,
    '.docx': parse_docx,
    '.hwpx': parse_hwpx,
    '.md': parse_md,
    '.txt': parse_txt,
    '.html': parse_html,
    '.pptx': parse_pptx,
    '.xlsx': parse_xlsx,
}

results: dict[pathlib.Path, ParsedDoc] = {}
skipped, failed = [], []

for path in FILES:
    parser = PARSERS.get(path.suffix.lower())
    if parser is None:
        skipped.append(path)
        continue
    try:
        results[path] = parser(path)
    except Exception as exc:
        failed.append((path, f'{type(exc).__name__}: {exc}'))

print(f'파싱 {len(results)}개 · 건너뜀 {len(skipped)}개 · 실패 {len(failed)}개')
for p in skipped:
    print('  건너뜀 :', p.relative_to(SAMPLES))
for p, err in failed:
    print('  실패   :', p.relative_to(SAMPLES), '→', err)

In [ ]:
print(f"{'파일':<44} {'블록':>4} {'쪽':>3} {'표':>3} {'글자수':>7}  비고")
print('-' * 80)
for path, d in results.items():
    chars = sum(len(b.text) for b in d.blocks)
    note = 'OCR 필요' if d.page_count and not d.blocks else ''
    name = str(path.relative_to(SAMPLES))
    print(f'{name[:44]:<44} {len(d.blocks):>4} {d.page_count:>3} {d.table_count:>3} {chars:>7,}  {note}')

## 12. 문서마다 첫 블록 살짝 보기

In [ ]:
for path, d in results.items():
    if not d.blocks:
        continue
    b = d.blocks[0]
    preview = b.text.replace('\n', ' ⏎ ')[:60]
    print(f'{path.name[:40]:<40} [{b.kind} · {b.locator}] {preview}')

## 13. manifest.yaml 과 맞춰 보기

- manifest 의 `file` 이 실제로 있고 파싱되었는지, 없으면 `fallback` 으로 대신할 수 있는지 확인

In [ ]:
import yaml

manifest = yaml.safe_load((SAMPLES / 'manifest.yaml').read_text(encoding='utf-8'))

for item in manifest['documents']:
    main = SAMPLES / item['file']
    fb = SAMPLES / item['fallback']
    if main in results:
        state = f"OK ({len(results[main].blocks)} 블록)"
    elif fb in results:
        state = f"fallback 사용 → {fb.name}"
    else:
        state = '파일 없음'
    print(f"{item['doc_id']} {item['version']:<5} {item['format']:<5} {state}")